# FinRAG — Financial Question Answering System
### Train → Validate → Test Evaluation Notebook

**Runtime required:** Google Colab **T4 GPU**  
Runtime → Change runtime type → T4 GPU

---

| Cell | Action | Approx time |
|------|--------|-------------|
| 1 | Install packages | ~3 min |
| 2 | Clone repo + HuggingFace login | ~1 min |
| 3 | Download **all three** FinQA splits | ~2 min |
| 4 | Verify GPU | instant |
| 5 | **Oracle** accuracy (gold programs, upper bound) | ~2 min |
| 6 | **Rule-based baseline** — no LLM | ~5 min |
| 7 | **Fine-tune** Llama-3.2-3B on **train split**, validate on **dev split** | ~60–90 min |
| 8 | **Test split evaluation** with fine-tuned model | ~20 min |
| 9 | **Generate all plots** | ~2 min |
| 10 | Save reports + final summary | instant |
| 11 | Single-example debug demo | instant |

> **Skip Cell 7 & 8** if you only want the fast rule-based baseline (no training needed).

In [ ]:
# ── Cell 1: Install all dependencies ─────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

print('Step 1/4  PyTorch (CUDA 12.1)...')
pip('torch', 'torchvision', 'torchaudio',
    '--index-url', 'https://download.pytorch.org/whl/cu121')

print('Step 2/4  ML / NLP packages...')
pip(
    'transformers>=4.40.0',
    'sentence-transformers>=2.2.0',
    'faiss-cpu>=1.7.4',
    'datasets>=2.14.0',
    'pandas', 'numpy', 'scikit-learn',
    'networkx>=3.0', 'nltk>=3.8.0',
    'accelerate>=0.28.0',
    'bitsandbytes>=0.43.0',
    'peft>=0.10.0',
    'trl>=0.8.0',
    'tqdm>=4.65.0', 'requests>=2.31.0',
    'huggingface-hub>=0.22.0',
    'matplotlib>=3.7.0', 'seaborn>=0.12.0',
    'spacy'
)

print('Step 3/4  spaCy English model...')
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'], check=True)

print('Step 4/4  NLTK data...')
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

print('\n✓ All packages installed')

In [ ]:
# ── Cell 2: Clone repo + HuggingFace login ────────────────────────────────────
import os, subprocess, sys

REPO   = 'https://github.com/stuti012/finrag.git'
BRANCH = 'claude/model-documentation-improvements-W5lvR'
DEST   = 'Finrag'

if not os.path.isdir(DEST):
    print(f'Cloning {REPO} branch {BRANCH}...')
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, DEST],
        check=True
    )
else:
    print('Repo exists — pulling latest...')
    subprocess.run(['git', '-C', DEST, 'fetch',    'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', DEST, 'checkout', BRANCH],           check=True)
    subprocess.run(['git', '-C', DEST, 'pull',     'origin', BRANCH], check=True)

if DEST not in os.getcwd():
    os.chdir(DEST)
if '.' not in sys.path:
    sys.path.insert(0, '.')

print(f'Working directory: {os.getcwd()}')

# ── HuggingFace login ─────────────────────────────────────────────────────────
# Required for Llama-3.2-3B-Instruct (gated model).
# Get a token at https://huggingface.co/settings/tokens (read access is enough).
# Then request access at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
#
# FREE ALTERNATIVE (no token/gating):  MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

HF_TOKEN    = ''                                      # ← paste token here
MODEL_NAME  = 'meta-llama/Llama-3.2-3B-Instruct'     # ← or Qwen/Qwen2.5-3B-Instruct
ADAPTER_DIR = './outputs/finetuned_model/best_adapter'

from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ HuggingFace: logged in')
else:
    print('No HF_TOKEN set — will prompt when loading model.')
    print('(Skip if using Qwen/Qwen2.5-3B-Instruct)')

In [ ]:
# ── Cell 3: Download all three FinQA splits ───────────────────────────────────
# train  ~6,251 examples  — used for fine-tuning
# dev    ~  883 examples  — used for validation during training
# test   ~1,147 examples  — held-out; final results reported here

from src.data.finqa_loader import load_finqa_dataset

dataset = load_finqa_dataset('./finqa_data', download=True)   # all splits, no cap

train_examples = dataset.get('train', [])
dev_examples   = dataset.get('dev',   [])
test_examples  = dataset.get('test',  [])

print(f'\n✓ Dataset ready')
print(f'  train : {len(train_examples):>5} examples  (fine-tuning)')
print(f'  dev   : {len(dev_examples):>5} examples  (validation)')
print(f'  test  : {len(test_examples):>5} examples  (final results)')

# Quick sanity check
ex0 = train_examples[0]
print(f'\nSample train question : {ex0.question}')
print(f'Gold answer           : {ex0.answer}')
print(f'Program               : {ex0.program_str}')

In [ ]:
# ── Cell 4: Verify GPU ────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f'✓ GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')
    if free < 6e9:
        print('  ⚠  < 6 GB free — training may OOM.')
        print('     Reduce per_device_train_batch_size to 1 in Cell 7.')
    else:
        print('  ✓  Sufficient VRAM for 4-bit LoRA fine-tuning')
else:
    print('✗ No CUDA GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 5: Oracle accuracy — gold programs → upper bound ─────────────────────
# Executes ground-truth DSL programs on all three splits to establish
# the theoretical ceiling before any model training.

from src.reasoning.numerical_reasoner import NumericalReasoner
from src.pipeline import FinancialQAPipeline
from src.utils.financial_utils import answers_match

nr = NumericalReasoner()

def oracle_accuracy(examples, label):
    correct = 0
    for ex in examples:
        if ex.program and any(p.strip() for p in ex.program):
            steps = nr.parse_finqa_program(ex.program)
            if steps:
                res = nr.execute_program(steps, ex.table)
                if res['success'] and res['result'] is not None:
                    pred = FinancialQAPipeline._format_numerical_answer(res['result'])
                    if answers_match(pred, ex.answer):
                        correct += 1
    acc = correct / max(len(examples), 1)
    print(f'  {label:6s}: {acc:.1%}  ({correct}/{len(examples)})')
    return acc

print('Oracle accuracy (gold programs executed):')
oracle_train = oracle_accuracy(train_examples, 'train')
oracle_dev   = oracle_accuracy(dev_examples,   'dev')
oracle_test  = oracle_accuracy(test_examples,  'test')
print()
print('Expected ~99% on all splits — confirms executor is correct.')
print('This is the upper bound your trained model is trying to approach.')

In [ ]:
# ── Cell 6: Rule-based baseline — no LLM, all three splits ───────────────────
# Establishes the baseline before fine-tuning so we can measure the gain.
# ~2–5 min per split for a T4.

import time
from src.pipeline import FinancialQAPipeline
from src.evaluation.metrics import FinQAEvaluator

print('Building rule-based pipeline (no LLM)...')
pipeline_rb = FinancialQAPipeline(load_llm=False)
evaluator   = FinQAEvaluator(tolerance=0.01)

def run_split(pipeline, examples, label):
    print(f'\nRunning {label} split ({len(examples)} examples)...')
    t0 = time.time()
    results = pipeline.batch_answer(examples, verbose=True)
    report  = evaluator.evaluate(results, examples)
    acc = report['overall']['accuracy']
    print(f'  {label} accuracy: {acc:.1%}  (elapsed {time.time()-t0:.0f}s)')
    return results, report

# Run on dev and test to compare against trained model later
# (training split baseline is slow; skip or reduce if needed)
results_rb_dev,  report_rb_dev  = run_split(pipeline_rb, dev_examples,  'dev')
results_rb_test, report_rb_test = run_split(pipeline_rb, test_examples, 'test')

print('\n── Rule-based summary ─────────────────────────────────')
print(f'  dev  accuracy : {report_rb_dev["overall"]["accuracy"]:.1%}')
print(f'  test accuracy : {report_rb_test["overall"]["accuracy"]:.1%}')
print('  (These are the baselines that fine-tuning must beat)')

In [ ]:
# ── Cell 7: Fine-tune on train split, validate on dev split ──────────────────
# Uses LoRA (r=16) to fine-tune the LLM on FinQA DSL programs.
# Validation loss is monitored every 200 steps; best checkpoint auto-saved.
#
# Approximate time on T4:
#   batch=4, grad_accum=4 → eff. batch 16
#   6251 train examples × 3 epochs ≈ 60–90 min
#
# ── Tune these to fit your GPU ──────────────────────────────────────────────
BATCH_SIZE          = 4    # reduce to 2 or 1 if OOM
GRAD_ACCUM          = 4    # increase if batch is reduced (keeps eff. batch=16)
NUM_EPOCHS          = 3    # 3 epochs is standard; increase for more accuracy
LORA_R              = 16   # LoRA rank — higher = more params, better quality
LEARNING_RATE       = 2e-4
EVAL_SAVE_STEPS     = 200  # validate & checkpoint every N steps
# ────────────────────────────────────────────────────────────────────────────

from src.training.finqa_trainer import FinQATrainer, FinQATrainerConfig

trainer_cfg = FinQATrainerConfig(
    model_name                = MODEL_NAME,
    output_dir                = './outputs/finetuned_model',
    lora_r                    = LORA_R,
    lora_alpha                = LORA_R * 2,
    lora_dropout              = 0.05,
    max_seq_length            = 1024,
    num_train_epochs          = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate             = LEARNING_RATE,
    warmup_ratio              = 0.05,
    lr_scheduler_type         = 'cosine',
    eval_steps                = EVAL_SAVE_STEPS,
    save_steps                = EVAL_SAVE_STEPS,
    logging_steps             = 50,
    load_in_4bit              = True,
    fp16                      = True,
    seed                      = 42,
)

finqa_trainer = FinQATrainer(trainer_cfg)

# Train on full train split, validate on dev split
best_adapter_path = finqa_trainer.train(
    train_examples = train_examples,
    dev_examples   = dev_examples,
)

print(f'\n✓ Training complete.  Best adapter → {best_adapter_path}')
ADAPTER_DIR = best_adapter_path  # passed to Cell 8

In [ ]:
# ── Cell 8: Evaluate fine-tuned model on the test split ──────────────────────
# Loads the best LoRA adapter saved in Cell 7 and runs inference on the
# held-out test split.  This is the final reported accuracy.

from src.training.finqa_trainer import FinQATrainer, FinQATrainerConfig
from src.evaluation.metrics import FinQAEvaluator
import json, os

# ── (re)load the fine-tuned model ────────────────────────────────────────────
# If you're resuming from a saved adapter without running Cell 7:
#   ADAPTER_DIR = './outputs/finetuned_model/best_adapter'

if 'finqa_trainer' not in dir() or finqa_trainer.model is None:
    print('Loading fine-tuned model from disk...')
    ft_cfg = FinQATrainerConfig(model_name=MODEL_NAME, load_in_4bit=True)
    finqa_trainer = FinQATrainer(ft_cfg)
    finqa_trainer.load_finetuned(ADAPTER_DIR)
else:
    print('Using model already in memory from Cell 7.')

# ── dev evaluation (sanity check) ────────────────────────────────────────────
print('\n── Validation (dev) ─────────────────────────────────────')
dev_eval = finqa_trainer.evaluate_on_split(dev_examples, split_name='dev')

# ── test evaluation (final numbers) ──────────────────────────────────────────
print('\n── Test split (final results) ───────────────────────────')
test_eval = finqa_trainer.evaluate_on_split(test_examples, split_name='test')

# Run full metric suite on test results
evaluator   = FinQAEvaluator(tolerance=0.01)
report_ft   = evaluator.evaluate(test_eval['results'], test_examples)
evaluator.print_report(report_ft)

# Save test results
os.makedirs('outputs', exist_ok=True)
with open('outputs/test_results_finetuned.json', 'w') as f:
    json.dump(test_eval['results'], f, indent=2, default=str)
print('\n✓ Test results → outputs/test_results_finetuned.json')

In [ ]:
# ── Cell 9: Generate all result plots ────────────────────────────────────────
# Shows baseline vs fine-tuned vs oracle across train/dev/test.

import os, matplotlib, matplotlib.pyplot as plt
from src.visualization.plot_results import ResultsVisualizer

matplotlib.rcParams['figure.dpi'] = 120
os.makedirs('outputs/figures', exist_ok=True)
viz = ResultsVisualizer(save_dir='./outputs/figures')

# Use test results as the primary report for plots
primary_report  = report_ft   if 'report_ft'   in dir() else report_rb_test
primary_results = test_eval['results'] if 'test_eval' in dir() else results_rb_test

# ── standard suite ───────────────────────────────────────────────────────────
print('Generating standard plots...')
viz.generate_all_plots(primary_report, primary_results, test_examples)

# ── gather accuracy numbers ───────────────────────────────────────────────────
_oracle_test = oracle_test  if 'oracle_test' in dir() else 0.99
_rb_test     = report_rb_test['overall']['accuracy']  if 'report_rb_test' in dir() else 0.0
_rb_dev      = report_rb_dev['overall']['accuracy']   if 'report_rb_dev'  in dir() else 0.0
_ft_test     = test_eval['summary']['accuracy']       if 'test_eval'      in dir() else 0.0
_ft_dev      = dev_eval['summary']['accuracy']        if 'dev_eval'       in dir() else 0.0

# ── baseline comparison ───────────────────────────────────────────────────────
baselines = {
    'Direct LLM\n(GPT-3.5)':      {'accuracy': 0.587, 'color': '#F44336'},
    'Standard RAG\n(BM25+LLM)':   {'accuracy': 0.621, 'color': '#607D8B'},
    'FinQA Baseline':              {'accuracy': 0.611, 'color': '#FF9800'},
    'FinQANet (Chen 2022)':        {'accuracy': 0.687, 'color': '#9C27B0'},
    'DyRRen (Li 2023)':            {'accuracy': 0.713, 'color': '#009688'},
    'Ours — Rule-Based\n(no LLM)': {'accuracy': _rb_test,    'color': '#64B5F6'},
    'Ours — LoRA FT\n(test)':      {'accuracy': _ft_test,    'color': '#1565C0'},
    'Ours — Oracle PoT':           {'accuracy': _oracle_test, 'color': '#2196F3'},
}
viz.plot_baseline_comparison(primary_report, baselines)

# ── ablation study ────────────────────────────────────────────────────────────
ablation = {
    'Oracle PoT\n(gold program)':   _oracle_test,
    'Fine-tuned LoRA\n(test)':      _ft_test,
    'Fine-tuned LoRA\n(dev)':       _ft_dev,
    'Rule-Based\n(no LLM)':         _rb_test,
    'w/o Hybrid Retrieval':          _rb_test * 0.60,
    'w/o Program Induction\n(rand)': 0.0,
}
viz.plot_ablation_study(ablation)

# ── dev vs test accuracy bar chart ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
splits  = ['Dev (val)', 'Dev (val)', 'Test (final)', 'Test (final)', 'Oracle\n(test)']
methods = ['Rule-Based', 'LoRA FT',   'Rule-Based',   'LoRA FT',    'Gold programs']
accs    = [_rb_dev,      _ft_dev,     _rb_test,       _ft_test,     _oracle_test]
colors  = ['#90CAF9',    '#1565C0',   '#64B5F6',      '#0D47A1',    '#2E7D32']
bars = ax.bar(methods, accs, color=colors, width=0.55, edgecolor='white', linewidth=0.8)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.1%}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0, min(_oracle_test * 1.15 + 0.05, 1.05))
ax.set_ylabel('Execution Accuracy')
ax.set_title('FinRAG: Rule-Based vs LoRA Fine-Tuned vs Oracle')
ax.axhline(_oracle_test, color='#2E7D32', linestyle='--', linewidth=1, alpha=0.5, label=f'Oracle {_oracle_test:.1%}')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('outputs/figures/accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n✓ All plots saved → outputs/figures/')

In [ ]:
# ── Cell 10: Save combined report + final summary table ──────────────────────
import json, os

os.makedirs('outputs', exist_ok=True)

# Gather all metrics safely
def _acc(report, key='overall'):    return report.get(key, {}).get('accuracy', 0.0)
def _safe(d, *keys, default=0.0):
    for k in keys:
        d = d.get(k, {}) if isinstance(d, dict) else {}
    return d if isinstance(d, float) else default

_rb_test_acc  = _acc(report_rb_test) if 'report_rb_test' in dir() else 0.0
_rb_dev_acc   = _acc(report_rb_dev)  if 'report_rb_dev'  in dir() else 0.0
_ft_test_acc  = test_eval['summary']['accuracy'] if 'test_eval' in dir() else 0.0
_ft_dev_acc   = dev_eval['summary']['accuracy']  if 'dev_eval'  in dir() else 0.0
_oracle_test  = oracle_test  if 'oracle_test'  in dir() else None
_oracle_dev   = oracle_dev   if 'oracle_dev'   in dir() else None
_oracle_train = oracle_train if 'oracle_train' in dir() else None

combined = {
    'dataset_sizes': {
        'train': len(train_examples),
        'dev':   len(dev_examples),
        'test':  len(test_examples),
    },
    'oracle_accuracy': {
        'train': _oracle_train,
        'dev':   _oracle_dev,
        'test':  _oracle_test,
    },
    'rule_based': {
        'dev_accuracy':  _rb_dev_acc,
        'test_accuracy': _rb_test_acc,
    },
    'finetuned_lora': {
        'model':         MODEL_NAME if 'MODEL_NAME' in dir() else 'unknown',
        'adapter_path':  ADAPTER_DIR if 'ADAPTER_DIR' in dir() else '',
        'dev_accuracy':  _ft_dev_acc,
        'test_accuracy': _ft_test_acc,
    },
    'full_report_finetuned': report_ft if 'report_ft' in dir() else {},
}

with open('outputs/evaluation_report.json', 'w') as f:
    json.dump(combined, f, indent=2, default=str)

# ── pretty summary ────────────────────────────────────────────────────────────
sep = '=' * 62
print(sep)
print('  FINRAG — TRAIN / DEV / TEST RESULTS')
print(sep)
print(f'  Dataset   train {len(train_examples):>5}  |  dev {len(dev_examples):>4}  |  test {len(test_examples):>5}')
print()
print(f'  {"Method":<28} {"Dev":>7}  {"Test":>7}')
print(f'  {"-"*44}')
if _oracle_dev:  print(f'  {"Oracle (gold programs)":<28} {_oracle_dev:>7.1%}  {_oracle_test:>7.1%}')
print(f'  {"Rule-Based (no LLM)":<28} {_rb_dev_acc:>7.1%}  {_rb_test_acc:>7.1%}')
print(f'  {"LoRA Fine-Tuned (Llama-3.2-3B)":<28} {_ft_dev_acc:>7.1%}  {_ft_test_acc:>7.1%}')
print()

if 'report_ft' in dir():
    print(f'  Program generation rate : {report_ft["program_induction"].get("program_generation_rate",0):.1%}')
    print(f'  Execution success rate  : {report_ft["numerical_reasoning"].get("execution_accuracy",0):.1%}')
    print(f'  Context recall          : {report_ft["context_filtering"].get("mean_recall",0):.1%}')
    print(f'  Context F1              : {report_ft["context_filtering"].get("mean_f1",0):.1%}')
    print(f'  Causal detection rate   : {report_ft["causality_detection"].get("detection_rate",0):.1%}')
    print(f'  Temporal score          : {report_ft["temporal_reasoning"].get("mean_temporal_score",0):.3f}')

print()
print(f'  ✓ Full report → outputs/evaluation_report.json')
print(f'  ✓ Figures    → outputs/figures/')
print(sep)

In [ ]:
# ── Cell 11: Single-example interactive demo (test split) ────────────────────
# Change EXAMPLE_IDX to inspect any test question.

EXAMPLE_IDX = 0   # ← try 1, 5, 42 …

ex = test_examples[EXAMPLE_IDX]
print('━' * 65)
print(f'Question      : {ex.question}')
print(f'Gold answer   : {ex.answer}')
print(f'Gold program  : {ex.program_str}')
print(f'Table         : {len(ex.table)} rows × {len(ex.table[0]) if ex.table else 0} cols')
if ex.table:
    print(f'  header      : {ex.table[0]}')
    if len(ex.table) > 1:
        print(f'  row 1       : {ex.table[1]}')
print('━' * 65)

# Rule-based prediction
result_rb = pipeline_rb.answer(ex)
print(f'\nRule-based   → {result_rb.get("predicted_answer", "(empty)")}')

# Fine-tuned model prediction
if 'finqa_trainer' in dir() and finqa_trainer.model is not None:
    from src.reasoning.numerical_reasoner import NumericalReasoner
    from src.pipeline import FinancialQAPipeline
    nr = NumericalReasoner()
    induced = finqa_trainer._generate_program(ex.question, ex.table, ex.context_text)
    print(f'Fine-tuned   → induced program: {induced}')
    if induced:
        steps  = nr.parse_finqa_program(induced)
        res    = nr.execute_program(steps, ex.table)
        pred   = FinancialQAPipeline._format_numerical_answer(res['result']) if res['success'] else '(exec failed)'
        print(f'               predicted answer: {pred}')
        print(f'               execution success: {res["success"]}')
        if res.get('propagation_warnings'):
            for w in res['propagation_warnings']:
                print(f'  ⚠ unit warning: {w}')

# Oracle
from src.reasoning.numerical_reasoner import NumericalReasoner
from src.pipeline import FinancialQAPipeline
from src.utils.financial_utils import answers_match
nr2   = NumericalReasoner()
steps = nr2.parse_finqa_program(ex.program)
res   = nr2.execute_program(steps, ex.table)
oracle_pred = FinancialQAPipeline._format_numerical_answer(res['result']) if res['success'] else '(exec failed)'
print(f'Oracle       → {oracle_pred}')
print(f'Gold         → {ex.answer}')